# Trim the data for Content Based Filtering.
Step 1 of 2. Next run the routines in Create Training.

In [ ]:
import re
import csv
import pandas as pd
import numpy as np
from numpy.random import default_rng
from collections import defaultdict

#doesn't match previous exactly
genre_names = ['(no genres listed)','Action','Adventure',
'Animation', "Children", "Comedy", "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", 
"Musical", "Mystery", "Romance", "Sci-Fi", "Thriller" , "War", "Western", "IMAX" ]
num_genre = len(genre_names)
print(num_genre)
min_rating_count = 10

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')


In [ ]:
#import os
#os.chdir('/content/drive/My Drive/Colab Notebooks/RecSys/data/parse')

In [ ]:
def get_movie_rating_count():
    """ just gets a count of ratings per movie from ratings file """
    count = 0
    movie_rating_count = defaultdict(int)
    with open('./ratings.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1 
                print(line)  # skip header
            else:
                count +=1
                movie_id = int(line[1])
                #if (movie_id == 105504): print("Speak up")
                rating = line[2]
                movie_rating_count[movie_id] += 1
        return movie_rating_count

In [ ]:
movie_rating_count = get_movie_rating_count()
105504 in movie_rating_count.keys()

In [ ]:
def loadMovieList():
    p = re.compile('^(.+)\s\((\d+)\)')
    count = 0
    data = []
    with open('./movies.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1
                print(line) 
            else:
                count +=1
                #if count > 5: break
                movie_id = int(line[0])   # note, no subract
                m = p.match(line[1])      # crack title
                if m == None: continue
                title = m.group(1)
                year = int(m.group(2))
                
                mgen = line[2].split("|")
                gv = [0] * num_genre
                for g in mgen:
                    idx = genre_names.index(g)
                    gv[idx] = 1
                data.append([movie_id, title, year, *gv])
        #print(len(data))
        movie_df = pd.DataFrame(data)
        movie_df.columns = ["movie id", "title", "year"] + genre_names
        return movie_df

In [ ]:
r0_df = loadMovieList()

Drop old movies

In [ ]:
r1_df = r0_df.copy(deep=True)
r1_df = r1_df[r1_df['year'] > 2000]  # 1692
print(len(r1_df.index))

Drop columns of underused genre

In [ ]:
r2_df = r1_df.copy(deep=True)
imaxes = r2_df[(r2_df['IMAX'] == 1) | (r2_df['(no genres listed)'] == 1)].index
r2_df.drop(imaxes,  axis=0, inplace=True)
r2_df.drop( 'IMAX', axis=1, inplace=True)
r2_df.drop( '(no genres listed)', axis=1, inplace=True)
print(len(r1_df.index))

### find number of ratings given genre type.

In [ ]:
for i in ['Action','Adventure',
'Animation', "Children", "Comedy", "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", 
"Musical", "Mystery", "Romance", "Sci-Fi", "Thriller" , "War", "Western" ]:
    count = len(r2_df[r2_df[i] == 1].index)
    print(i, count)

In [ ]:
r3_df = r2_df.copy(deep=True)
todrop = r3_df[(r3_df['War'] == 1) | (r3_df['Western'] == 1) | (r3_df['Musical'] == 1) | (r3_df['Film-Noir'] == 1)].index
r3_df.drop(todrop, axis=0, inplace=True)
print(len(r3_df.index))
r3_df.drop( 'War',       axis=1, inplace=True)
r3_df.drop( 'Western',   axis=1, inplace=True)
r3_df.drop( 'Musical',   axis=1, inplace=True)
r3_df.drop( 'Film-Noir', axis=1, inplace=True)
print(r3_df.columns)

In [ ]:
# remove movies with too few ratings, but some ratings
indexes = []
r4_df = r3_df.copy(deep=True)
for mid in movie_rating_count.keys():
    if movie_rating_count[mid] < min_rating_count:
        indx = r4_df[r4_df['movie id'] == mid ].index
        if not indx.empty:
            r4_df.drop(indx, inplace=True)
print(len(r4_df.index))

In [ ]:
# remove movies with no ratings (how did these get in here?, they are in rating file.)
r5_df = r4_df.copy(deep=True)
for indx, row in r5_df.iterrows():
    mid = row["movie id"]
    if mid not in movie_rating_count.keys():
        print(f"movie without rating: {mid}")
        r5_df.drop(indx, inplace=True)
print(len(r5_df.index))

In [ ]:
#maybenot = [4255,5151, 5225, 5617, 7318, 7346, 34338, 41571, 61250, 62434]
#r5_df[r5_df["movie id"].isin(maybenot) == True]

In [ ]:
maybenot = [4255,5151, 5225, 5617, 7318, 7346, 34338, 41571, 61250, 62434]
r6_df = r5_df.copy(deep=True)
todrop = r6_df[r6_df["movie id"].isin(maybenot) == True].index
r6_df.drop(todrop, inplace=True)
print(len(r6_df.index))

In [ ]:
movie_df = r6_df.copy(deep=True)
movie_df.reset_index(drop=True, inplace=True)
movie_df

In [ ]:
movie_df.loc[movie_df['movie id'] == 85564]  # checking movie that has no rating

# Users and ratings


In [ ]:
def loadUserRating():
    """ read ratings.csv and put into rating_df.
    only put in movies  in movie_dict 
    """
    # get all the movies, uniquify (required?), put in a ditionary for use in checking  later.  ( a bit dubious)
    movie_dictionary = dict.fromkeys(movie_df['movie id'].tolist())  # for checking if in movie_df
    p = re.compile('^(.+)\s\((\d+)\)')
    count = 0
    data = []
    with open('./ratings.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count +=1 
                print(line)  # skip header
            else:
                count +=1
                #if count > 5: break
                user_id = int(line[0])
                movie_id = int(line[1]) 
                rating = line[2]
                if movie_id not in movie_dictionary: continue
                data.append([user_id, movie_id, rating])
        rating_df = pd.DataFrame(data)
        print(f"number of ratings that have movies in movies_df: {len(rating_df.index)}")
        rating_df.columns = ["user id", "movie id", "rating"] 
        return rating_df

In [ ]:
rating_df = loadUserRating()

In [ ]:
uid_rating_df = rating_df.copy(deep=True)

In [ ]:
rating_df.to_csv('post_trim_rating_df.csv')
movie_df.to_csv('post_trim_movie_df.csv')

Should leave with ratings of movies in movies_df and movies with ratings in rating_df.